# Aery Jones et al. (2021) from dandiset 000165: linear track (run) sessions

**Paper.** Aery Jones EA, Rao A, Zilberter M, Djukic B, Bant JS, Gillespie AK, Koutsodendris N, Nelson M,
Yoon SY, Huang K, Yuan H, Gill TM, Huang Y, Frank LM (2021). Dentate gyrus and CA3 GABAergic
interneurons bidirectionally modulate signatures of internal and external drive to CA1. *Cell
Reports* 37, 110159. [doi:10.1016/j.celrep.2021.110159](https://doi.org/10.1016/j.celrep.2021.110159).
The claims file (`claims/jones-2021.yaml`) was written from the accepted author manuscript in PubMed
Central (PMC9069800) with its main supplement; Table S7 was not available to the reader.

**Dandiset.** [DANDI:000165](https://dandiarchive.org/dandiset/000165/0.211118.1526), version
0.211118.1526 (doi:10.48324/dandi.000165/0.211118.1526). The paper links the draft, not a version.
This is the only published version, and the draft has the same 572 assets and size, so the most
recent published version is pinned. Files are streamed with remfile and h5py; nothing is written to disk.

**Notebooks.** `rest_home_cage.ipynb` tests the claims on the home cage (rest) sessions;
`run_linear_track.ipynb` tests those on the linear track (run) sessions. Methods are in the helper
`jones2021.py`, whose docstring says where each parameter comes from.

**Common method (Method M of the claims file).** For each feature, a linear mixed model feature ~
treatment + (treatment|animal), fitted by maximum likelihood, with an F test of the treatment
coefficient (CNO minus vehicle); the model is fitted exactly from per-mouse, per-treatment sufficient
statistics (cell counts, means and sums of squares), which gives the same likelihood as fitting every
observation. The genotype test is the likelihood ratio of feature ~ treatment + (treatment|animal)
against feature ~ treatment:genotype + (treatment:genotype|animal) on the pooled data, with 9 degrees
of freedom as Table S3 gives. Two readings of the log transform are run: (a) log when the within-cell
residuals have skewness above 1 (assumption: the Methods log right-skewed features without a rule);
(b) log for every feature (the code). Significance is Holm-Bonferroni at 0.05 over the paper's family
(24 rest or 36 run comparisons per group or pair), with comparisons not computed here ranked after
those computed.

| Claim | Tested in | Status |
|---|---|---|
| fig1-tables1-expression | none | blocked: data_missing (histology not deposited) |
| figs1ef-exvivo-cno | none | blocked: data_missing (slice recordings not deposited) |
| fig2ef-csd-sr | none | blocked: other (cohort 1 is in the dandiset; not run within this run's time) |
| fig3b-mua-during-swr | rest | tested |
| fig3c-swr-rate-bidirectional | rest | tested |
| fig3d-swr-chains | rest | tested |
| fig3e-swr-length-sst | rest | tested |
| swr-longer-100ms-sst | rest | tested |
| fig3f-swr-frequency-pv | rest | tested |
| fig3g-sw-amplitude-pv | rest | tested |
| figs2a-swr-size-pv | rest | tested |
| tables2-fast-ripples-pv | none | blocked: data_missing (LFP filtered at 300 Hz, sampled at 1 kHz) |
| tables2-detection-unchanged | rest | tested |
| tables2-rest-sg-pv | rest | tested |
| fig4d-swr-sg-power-pv | none | blocked: other (SWR-triggered spectra not run within this run's time) |
| fig4e-swr-sg-coherence-pv | none | blocked: other (as fig4d) |
| figs2cd-sg-predicts-sw-frequency | none | blocked: other (depends on fig4d) |
| ca3-swr-overlap | rest | tested |
| figs3-ca3-swr-pv | rest | tested |
| figs3c-ca3-swr-rate-sst | rest | tested |
| figs3-dentate-spikes | rest | tested |
| fig5b-mua-run to figs5-theta-coherence-frequency (14 claims) | run | tested |
| running-speed-unchanged | run | tested |
| tables4-genotype-baseline | run | tested |
| tables6-sex-swr-size | rest and run | tested |
| sex-stratified-effects | rest and run | tested |
| figs6-double-like-pv, figs6-double-unlike-sst, figs6-empty-vector-control | rest and run | tested |

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
from scipy import stats

import jones2021 as J

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 400)
started = time.time()

assets = J.list_assets()
lt_assets = assets[assets.kind == "LinearTrack"]
print(f"dandiset {J.DANDISET_ID} version {J.DANDISET_VERSION}: {len(lt_assets)} linear track files of {len(assets)}")
results = J.process_many(lt_assets.asset_id.tolist(), "LinearTrack", workers=4)
errors = [r for r in results if "error" in r]
print(f"processed in {(time.time() - started) / 60:.1f} min; failed: {len(errors)}")
for e in errors:
    print("  ", e)

## Data check

In [ ]:
S = J.session_table(results)
S = S[S.animal.notna()].reset_index(drop=True)
chk = pd.DataFrame([dict(session=r["meta"]["session_id"], probe=r["meta"]["probe"], lfp_min=r["lfp_range"][0],
                         lfp_max=r["lfp_range"][1], conversion=r["conv"], pos_hz=r["pos"]["pos_hz"],
                         pos_end_minus_lfp_end=r["pos"]["pos_t1"] - r["n_samples"] / 1000,
                         x_range=np.ptp(r["pos"]["x_range"]), y_range=np.ptp(r["pos"]["y_range"]),
                         run_min=r["run_s"] / 60, bins_1s=r["n_bins"], speed_run=r["speed_run"], speed_all=r["speed_all"],
                         zero_spikes=r["n_zero_spikes"], n_ca1_pyr=len(r["regions"]["ca1_pyr"]), n_ca3=len(r["regions"]["ca3"]),
                         n_dg=len(r["regions"]["dg"]), clock_m30=r["clock"][-30], clock_0=r["clock"][0], clock_p30=r["clock"][30],
                         seconds=r["seconds"], maxrss_gb=r["maxrss_kb"] / 1e6)
                    for r in (results[i] for i in S.i)])
print("Probes:", chk.probe.unique())
print("Mice and sessions by group and treatment:")
print(S.groupby(["group", "treatment"]).agg(sessions=("session", "size"), mice=("animal", "nunique"),
                                             minutes=("minutes", "median")).round(1).to_string())
seq = S.assign(n=S.session.str.extract(r"(\d+)_")[0].astype(int)).sort_values(["animal", "n"]).groupby("animal").agg(
    group=("group", "first"), sequence=("treatment", lambda t: "".join("C" if x == "CNO" else "v" for x in t)),
    sessions=("n", lambda s: ",".join(map(str, s))))
print("\nTreatment sequence per mouse (v vehicle, C CNO):")
print(seq.to_string())
print("\nPer-session values:")
print(chk.drop(columns=["session", "probe"]).describe().round(3).T.to_string())
st = S.assign(start=[pd.Timestamp(results[i]["meta"]["start"]) for i in S.i]).sort_values(["animal", "start"])
gap = st.groupby("animal").start.apply(lambda t: t.diff().min().total_seconds() / 60)
print("\nShortest gap between the starts of two track sessions of one mouse (min):")
print(gap.describe().round(1).to_string())
print("Mice with two session starts less than 30 min apart:", int((gap < 30).sum()), "of", len(gap))
print("\nSessions without CA1-pyr, CA3 or DG sites:", int((chk.n_ca1_pyr == 0).sum()), int((chk.n_ca3 == 0).sum()), int((chk.n_dg == 0).sum()))

Checks of the data:

- 170 linear track files from 30 mice (8 PV-Cre, 7 SST-Cre, 7 PV-Cre/SST-Cre, 8 empty vector), all on
  the 4-shank probe, as the paper's cohort 3. Sessions last about 25 min; the Methods say 30 min.
- Every mouse has vehicle sessions first and CNO sessions after them (sequence column); the Methods say
  the treatments alternated. Treatment is confounded with session order.
- The session start times put the track sessions of each mouse minutes apart (shortest gap above), which
  cannot be right for 25 min sessions; the start times are wrong.
- The track is about 37 cm long in x (x_range), as the Methods' 36 cm track. Position is at 19.3 Hz.
- LFP values stay within about +-12500 file units on the track. As in the home cage notebook, the unit
  attribute says microvolts.

In [ ]:
print("Clock check: correlation over 1 s bins of CA1 log theta power with speed, speed shifted by -30, 0 and +30 s")
print(chk[["clock_m30", "clock_0", "clock_p30"]].describe().round(3).to_string())
print("Sessions where the unshifted correlation is the largest of the three:",
      int(((chk.clock_0 > chk.clock_m30) & (chk.clock_0 > chk.clock_p30)).sum()), "of", len(chk))
print("Peak memory of the workers (GB):", round(chk.maxrss_gb.max(), 2), "; mean seconds per file:", round(chk.seconds.mean(), 1))

Clock of position against the LFP. CA1 theta power is expected to rise with running speed. Over 1 s
bins the correlation of log theta power with speed is no larger at the true alignment (mean 0.030) than
with the speed shifted by 30 s either way (0.037 and 0.052), and the true alignment gives the largest of
the three in 79 of 170 sessions. This check does not establish that position and LFP share a clock on the
track; it is not evidence against it either, as the speed range on a 37 cm track is small. The run
periods are taken from the deposited times as they are.

In [ ]:
GROUPS = ("PV", "SST", "PS", "EV")
PAIRS = (("PV", "SST"), ("PV", "PS"), ("SST", "PS"))
t0 = time.time()
TAB = J.build_tables(results, S, J.RUN_FEATURES)
res = J.test_all(TAB, groups=GROUPS, pairs=PAIRS)
res = J.add_decisions(res, J.RUN_FAMILY, list(J.RUN_FEATURES))
print(f"Run comparisons: {len(J.RUN_FEATURES)} features (the paper's {J.RUN_FAMILY}); tests run: {len(res)} ({time.time() - t0:.0f} s)")
print("Observations per feature:", {k: len(v) for k, v in TAB.items()})
print(res[res.reading == "a"].pivot_table(index="feature", columns="test", values="p", aggfunc="first").map(lambda v: f"{v:.2g}").to_string())

def part_table(parts, r=None):
    t = J.show_parts(res if r is None else r, parts)
    print(t.to_string(index=False, float_format=lambda v: f"{v:.3g}"))

## running-speed-unchanged

> Running speed does not differ between vehicle and CNO sessions (4.8 against 5.1 cm/s).

Paired t test across the track mice of each mouse's mean speed per treatment (mean over sessions).
Readings: (a) smoothed speed over run (above 1 cm/s); (b) over the whole session.

In [ ]:
D = pd.DataFrame([dict(animal=a, group=g, cno=int(tr == "CNO"), run=results[i]["speed_run"], whole=results[i]["speed_all"])
                  for i, a, g, tr in zip(S.i, S.animal, S.group, S.treatment)])
for col, name in (("run", "(a) run periods"), ("whole", "(b) whole session")):
    m = D.groupby(["animal", "cno"])[col].mean().unstack().dropna()
    tt = stats.ttest_rel(m[1], m[0])
    print(f"{name}: {len(m)} mice, vehicle {m[0].mean():.2f} cm/s, CNO {m[1].mean():.2f} cm/s, "
          f"t({len(m) - 1}) = {tt.statistic:.2f}, p = {tt.pvalue:.3g}; "
          f"within tolerance: {abs(m[0].mean() - 4.8) <= 0.48 and abs(m[1].mean() - 5.1) <= 0.51}")
print("Range of mouse means (a):", round(D.run.min(), 2), "to", round(D.run.max(), 2), "cm/s")

**Result: not reproduced.** Running speed during run is 4.70 cm/s on vehicle days and 5.24 cm/s on CNO days, both within tolerance of 4.8 and 5.1, but the paired t test across 30 mice is significant (t(29) = 2.69, p = 0.012), where the paper found p = 0.2. Over the whole session the difference is larger (0.96 against 1.51 cm/s, p = 8.4e-06). With vehicle days always first, this can be an effect of practice.

## fig5b-mua-run

> During movement, suppressing either class increases MUA in DG and CA3 but not in CA1.

MUA rate in 100 ms bins wholly in run, at each region's MUA site. Reading (a) logs right-skewed features; (b) logs all.

In [ ]:
part_table([("mua_ca1", "PV", "ns"), ("mua_ca1", "SST", "ns"), ("mua_ca3", "PV", "up"), ("mua_ca3", "SST", "up"),
                ("mua_dg", "PV", "up"), ("mua_dg", "SST", "up")])

**Result: not reproduced.** MUA during run does not change in CA1 (p = 0.14 and 0.046, not significant after correction) and does not increase in CA3 or DG in either genotype; all four CA3 and DG coefficients are negative (p from 0.09 to 0.82). Reading (b) gives the same CA3 and DG result and makes the SST-Cre CA1 decrease significant.

## fig5d-sg-power-run

> During movement, suppressing PV+ interneurons amplifies SG power in CA1 with no effect in DG and CA3, while suppressing SST+ interneurons dampens SG power in DG and CA3 with no effect in CA1; all these effects differ between genotypes.

In [ ]:
part_table([("sg_pow_ca1", "PV", "up"), ("sg_pow_ca3", "PV", "ns"), ("sg_pow_dg", "PV", "ns"),
                ("sg_pow_ca1", "SST", "ns"), ("sg_pow_ca3", "SST", "down"), ("sg_pow_dg", "SST", "down"),
                ("sg_pow_ca1", "PV vs SST", "sig"), ("sg_pow_ca3", "PV vs SST", "sig"), ("sg_pow_dg", "PV vs SST", "sig")])

**Result: not reproduced.** SST+ suppression dampens SG power in CA3 (p = 0.00064) and DG (p = 1.2e-06) and leaves CA1 unchanged (p = 0.028, not significant after correction), and PV+ suppression leaves CA3 and DG unchanged, as claimed. But PV+ suppression does not amplify SG power in CA1 (coefficient -0.10, p = 0.71), and no genotype test is significant (p = 0.19, 0.45, 0.44). Five of nine parts hold; the partial rule needs the CA1 PV-Cre increase.

## fig5e-sg-coherence-run

> During movement, suppressing PV+ interneurons enhances SG coherence between CA1 and CA3.

Readings of the bins: (a) coherence per 1 s bin (the main table); (b) one coherence per session over all
run, tested with the same LMM on session values.

In [ ]:
part_table([("sg_coh_ca3", "PV", "up")])
print("\nReported: SST-Cre and genotype test")
part_table([("sg_coh_ca3", "SST", "ns"), ("sg_coh_ca3", "PV vs SST", "ns")])
SESS = {}
for k in ("theta_coh_ca3", "sg_coh_ca3", "fg_coh_ca3", "theta_coh_dg", "sg_coh_dg", "fg_coh_dg"):
    SESS[k] = J.long_table(results, S, lambda r, k=k: [r["coh_session"].get(k, np.nan)])
res_sess = J.add_decisions(J.test_all(SESS, groups=("PV", "SST"), pairs=(("PV", "SST"),)), J.RUN_FAMILY, list(SESS))
print("\nReading (b), session coherence (decisions with these six as the computed family):")
part_table([("sg_coh_ca3", "PV", "up"), ("sg_coh_ca3", "SST", "ns")], res_sess)

**Result: not reproduced.** SG coherence between CA1 and CA3 decreases with PV+ suppression (per 1 s bin -0.034, p = 0.0024, not significant after Holm untransformed and significant logged; over the session -0.077, p = 0.00024). The claim is an increase.

## fig5f-sg-frequency-pv

> Suppressing PV+, but not SST+, interneurons reduces SG frequency during movement in CA1, CA3 and DG, and the genotypes differ.

In [ ]:
part_table([("sg_freq_ca1", "PV", "down"), ("sg_freq_ca3", "PV", "down"), ("sg_freq_dg", "PV", "down"),
                ("sg_freq_ca1", "SST", "ns"), ("sg_freq_ca3", "SST", "ns"), ("sg_freq_dg", "SST", "ns")])
print("\nGenotype tests, reported:")
part_table([("sg_freq_ca1", "PV vs SST", "sig"), ("sg_freq_ca3", "PV vs SST", "sig"), ("sg_freq_dg", "PV vs SST", "sig")])

**Result: not reproduced.** PV+ suppression lowers SG frequency in CA3 (p = 0.00015) but not significantly in CA1 (p = 0.30) or DG (p = 0.13); SST+ suppression changes it nowhere (p from 0.42 to 0.63). The partial rule needs the CA1 decrease. No genotype test is significant (p from 0.18 to 0.43). 7 PV-Cre mice have a CA3 frequency site and 5 SST-Cre mice a DG one.

## fig5g-sg-theta-modulation

> Suppressing PV+ interneurons increases theta modulation of SG locally in DG and CA3 but not in CA1, while suppressing SST+ interneurons reduces theta modulation of SG in CA1, and these effects differ between genotypes.

In [ ]:
part_table([("sg_mi_ca3", "PV", "up"), ("sg_mi_dg", "PV", "up"), ("sg_mi_ca1", "SST", "down"), ("sg_mi_ca1", "PV", "ns"),
                ("sg_mi_ca3", "PV vs SST", "sig"), ("sg_mi_dg", "PV vs SST", "sig")])
print("\nCA1 genotype test, reported:")
part_table([("sg_mi_ca1", "PV vs SST", "ns")])

**Result: not reproduced.** Theta modulation of SG decreases with PV+ suppression in CA3 (p = 0.018) and DG (p = 0.00099, significant), the opposite of the claim, and does not decrease in CA1 with SST+ suppression (coefficient +0.02, p = 0.56). The PV-Cre CA1 effect is not significant (p = 0.11), as claimed, and the CA3 and DG genotype tests are not (p = 0.51 and 0.51).

## fig6b-fg-power-sst

> Suppressing SST+ interneurons amplifies FG power in CA1 and DG during movement, an effect specific to this class.

In [ ]:
part_table([("fg_pow_ca1", "SST", "up"), ("fg_pow_dg", "SST", "up"), ("fg_pow_ca1", "PV", "ns"), ("fg_pow_dg", "PV", "ns"),
                ("fg_pow_ca1", "PV vs SST", "sig"), ("fg_pow_dg", "PV vs SST", "sig")])

**Result: not reproduced.** FG power does not rise with SST+ suppression: it falls in CA1 (p = 0.080) and falls significantly in DG (p = 1.1e-05). PV-Cre effects are not significant (p = 0.48 and 0.0099 in CA1 and DG), and neither genotype test is (p = 0.54 and 0.69).

## figs4e-sg-fg-inverse

> In DG and CA3 of SST-Cre mice, the CNO-induced change in SG power predicts the change in FG power, animals with the greatest SG suppression showing the greatest FG enhancement.

Per SST-Cre mouse, CNO minus vehicle; (a) difference of the mouse's means (log scale where the
feature is logged under reading (a) of the log transform, and raw); (b) difference of the LMM fitted
conditional responses. Pearson correlation across mice. Expected R2 0.81 (CA3) and 0.84 (DG), tolerance 10%.

In [ ]:
rows = []
for reg in ("ca3", "dg"):
    for scale in ("raw", "log"):
        dsg, dfg = TAB[f"sg_pow_{reg}"].copy(), TAB[f"fg_pow_{reg}"].copy()
        if scale == "log":
            dsg["value"], dfg["value"] = J.log_values(dsg.value), J.log_values(dfg.value)
        for reading, fn in (("a mean difference", J.mean_changes), ("b LMM conditional", J.blup_changes)):
            x, y = fn(dsg, "SST"), fn(dfg, "SST")
            x, y = x.align(y, join="inner")
            r, p = stats.pearsonr(x, y)
            rows.append(dict(region=reg, scale=scale, reading=reading, mice=len(x), r=r, R2=r * r, p=p,
                             sg_change_range=f"{x.min():.3g} to {x.max():.3g}", fg_change_range=f"{y.min():.3g} to {y.max():.3g}"))
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**Result: not reproduced.** Across the 7 SST-Cre mice the CNO-induced changes in SG and FG power are positively correlated, not negatively: in CA3 r = 0.92 (raw mean differences, p = 0.0034; logged r = 0.83, p = 0.020) and in DG r = 0.81 (p = 0.027; logged r = 0.67, p = 0.10). Both readings of a mouse's change give the same values. The mice with the largest SG decrease have the largest FG decrease.

## fig6e-sg-fg-ratio

> SST+ suppression decreases the SG/FG power ratio in CA1, PV+ suppression increases it throughout the hippocampus, and these effects differ between genotypes across the hippocampus.

In [ ]:
part_table([("sgfg_ca1", "SST", "down"), ("sgfg_ca1", "PV", "up"), ("sgfg_ca3", "PV", "up"), ("sgfg_dg", "PV", "up"),
                ("sgfg_ca1", "PV vs SST", "sig"), ("sgfg_ca3", "PV vs SST", "sig"), ("sgfg_dg", "PV vs SST", "sig")])

**Result: not reproduced.** PV+ suppression raises the SG/FG ratio in CA3 (p = 0.00055) and DG (p = 0.00087), as claimed, but not in CA1 (p = 0.32), and SST+ suppression does not lower it in CA1 (p = 0.14). No genotype test is significant (p = 0.17, 0.040 and 0.051). The partial rule needs the CA1 parts.

## fig6f-fg-coherence-unchanged

> FG coherence between CA1 and CA3 and between CA1 and DG does not change with either suppression.

In [ ]:
part_table([("fg_coh_ca3", "PV", "ns"), ("fg_coh_ca3", "SST", "ns"), ("fg_coh_dg", "PV", "ns"), ("fg_coh_dg", "SST", "ns")])
print("\nReading (b), session coherence:")
part_table([("fg_coh_ca3", "PV", "ns"), ("fg_coh_ca3", "SST", "ns"), ("fg_coh_dg", "PV", "ns"), ("fg_coh_dg", "SST", "ns")], res_sess)

**Result: reproduced.** FG coherence does not change between CA1 and CA3 or CA1 and DG with either suppression: per 1 s bin p from 0.20 to 0.50, and over each session p from 0.23 to 0.90, with both log readings. The coefficients are -0.02 to +0.03.

## fig6g-fg-frequency-sst

> Suppressing SST+ interneurons increases FG frequency throughout the hippocampus.

In [ ]:
part_table([("fg_freq_ca1", "SST", "up"), ("fg_freq_ca3", "SST", "up"), ("fg_freq_dg", "SST", "up")])

**Result: partially reproduced.** FG frequency rises with SST+ suppression in DG (p = 1.5e-06, 5 mice with a DG frequency site) but not significantly in CA1 (p = 0.81) or CA3 (p = 0.24): one region of three.

## fig6h-fg-theta-modulation-sst

> Suppressing SST+ interneurons increases theta modulation of FG in CA1, an effect specific to SST+ interneurons.

In [ ]:
part_table([("fg_mi_ca1", "SST", "up"), ("fg_mi_ca1", "PV vs SST", "sig")])
print("\nPV-Cre, reported with its sign:")
part_table([("fg_mi_ca1", "PV", "down")])

**Result: not reproduced.** Theta modulation of FG in CA1 does not increase significantly with SST+ suppression (coefficient 0.073, p = 0.085), and the genotype test is not significant (p = 0.15). The PV-Cre effect is negative (-0.14, p = 0.067).

## figs4h-fg-theta-modulation-pv

> Suppressing PV+ interneurons dampens theta modulation of FG during movement.

In [ ]:
part_table([("fg_mi_ca3", "PV", "down"), ("fg_mi_dg", "PV", "down")])
print("\nCA1, reported with its sign:")
part_table([("fg_mi_ca1", "PV", "down")])

**Result: partially reproduced.** PV+ suppression dampens theta modulation of FG in DG (p = 2.1e-06) and in CA3 with p = 0.0098, which does not pass Holm; in CA1 the effect is negative too (p = 0.067). The claim holds in one of the two required regions.

## figs5-theta-power-ca1

> Suppressing either PV+ or SST+ interneurons increases theta power in CA1 but not in CA3 or DG during movement.

In [ ]:
part_table([("theta_pow_ca1", "PV", "up"), ("theta_pow_ca1", "SST", "up"), ("theta_pow_ca3", "PV", "ns"),
                ("theta_pow_ca3", "SST", "ns"), ("theta_pow_dg", "PV", "ns"), ("theta_pow_dg", "SST", "ns")])

**Result: not reproduced.** Theta power in CA1 does not rise with either suppression (PV-Cre -0.34 on the log scale, p = 0.34; SST-Cre -0.07, p = 0.80). In DG it falls with SST+ suppression (p = 0.0015, significant) and with PV+ suppression (p = 0.0019, not significant after Holm).

## figs5-theta-coherence-frequency

> Neither interneuron type modifies theta coherence or theta frequency, except for an increase in theta frequency in DG of PV-Cre mice.

In [ ]:
parts = [("theta_freq_dg", "PV", "up")] + [(f, g, "ns") for f in ("theta_coh_ca3", "theta_coh_dg", "theta_freq_ca1", "theta_freq_ca3", "theta_freq_dg")
                                           for g in ("PV", "SST") if not (f == "theta_freq_dg" and g == "PV")]
part_table(parts)
print("\nCoherence reading (b), session coherence:")
part_table([(f, g, "ns") for f in ("theta_coh_ca3", "theta_coh_dg") for g in ("PV", "SST")], res_sess)

**Result: not reproduced.** Theta frequency rises in DG with PV+ suppression (p = 1.1e-07), the stated exception, but it also rises significantly in CA3 (p = 0.0011), which the claim excludes. The other eight tests are not significant (theta coherence p from 0.015 to 0.47; frequency p from 0.012 to 0.92).

## tables4-genotype-baseline

> During vehicle treatment, PV-Cre mice differ from SST-Cre mice in DG and CA3 SG frequency (lower) and SG/FG ratios (higher).

Vehicle sessions of PV-Cre and SST-Cre mice; LMM feature ~ genotype + (genotype|animal) with an F test
(the same fit as the treatment test, with genotype in place of treatment); group means are means over
mice of each mouse's mean. Holm over the 36 run features.

In [ ]:
rows = []
for f in J.RUN_FEATURES:
    d = TAB[f]
    d = d[d.group.isin(("PV", "SST")) & d.session.isin(S[S.treatment == "Vehicle"].session)].copy()
    d["cno"] = (d.group == "SST").astype(int)          # coefficient is SST minus PV
    log = J.within_skew(d) > J.SKEW_LOG
    dd = d.copy()
    if log:
        dd["value"] = J.log_values(dd.value)
    r = J.treatment_test(J.cell_stats(dd, "value"))
    m = d.groupby(["group", "animal"]).value.mean().groupby(level=0).mean()
    rows.append(dict(feature=f, log=log, sst_minus_pv=r["coef"], p=r["p"], PV=m["PV"], SST=m["SST"]))
G4 = pd.DataFrame(rows)
G4["holm_36"] = J.holm(G4.p.values, 36)
show = G4[G4.feature.isin(["sg_freq_ca3", "sg_freq_dg", "sgfg_ca3", "sgfg_dg"])].copy()
exp = {"sg_freq_ca3": (40.54, 41.9), "sg_freq_dg": (41.06, 42.4), "sgfg_ca3": (6.2, 3.99), "sgfg_dg": (5.16, 3.4)}
show["expected_PV"] = show.feature.map(lambda f: exp[f][0])
show["expected_SST"] = show.feature.map(lambda f: exp[f][1])
print(show.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("\nOther run features with a vehicle genotype difference under Holm:", G4[G4.holm_36 & ~G4.feature.isin(list(exp))].feature.tolist())

**Result: partially reproduced.** In vehicle sessions PV-Cre mice have lower CA3 SG frequency (39.5 against 41.0 Hz, p = 0.0012) and higher SG/FG ratios in CA3 (6.08 against 3.89, p = 0.00054) and DG (5.09 against 3.27, p = 0.00026), each passing Holm over 36; DG SG frequency is lower in PV-Cre mice (40.3 against 41.2 Hz) but p = 0.038 does not pass. All eight group means are within tolerance of Table S4. Three of the four tests hold.

## tables6-sex-swr-size and sex-stratified-effects (run features)

Run features for the two sex claims: the vehicle sex difference (all track mice) and, per group,
the treatment by sex likelihood ratio test.

In [ ]:
rows, rows2 = [], []
Sv = S[S.treatment == "Vehicle"]
for f in J.RUN_FEATURES:
    d = TAB[f][TAB[f].session.isin(Sv.session)].copy()
    d["cno"] = (d.sex == "F").astype(int)
    log = J.within_skew(d) > J.SKEW_LOG
    if log:
        d["value"] = J.log_values(d.value)
    r = J.treatment_test(J.cell_stats(d, "value"))
    rows.append(dict(feature=f, log=log, female_minus_male=r["coef"], p=r["p"]))
    for g in ("PV", "SST", "PS"):
        d = TAB[f][TAB[f].group == g].copy()
        if J.within_skew(d) > J.SKEW_LOG:
            d["value"] = J.log_values(d.value)
        csF, csM = J.cell_stats(d[d.sex == "F"], "value"), J.cell_stats(d[d.sex == "M"], "value")
        if csF.animal.nunique() < 2 or csM.animal.nunique() < 2:
            continue
        gt = J.genotype_test(csF, csM)
        rows2.append(dict(group=g, feature=f, coef_F=gt["coef_a"], coef_M=gt["coef_b"], p=gt["p_df9"],
                          mice_F=csF.animal.nunique(), mice_M=csM.animal.nunique()))
SEXR = pd.DataFrame(rows)
SEXR["holm_36"] = J.holm(SEXR.p.values, 36)
SEXR["p<0.05"] = SEXR.p < 0.05
print("Vehicle sex differences, run features:")
print(SEXR.sort_values("p").to_string(index=False, float_format=lambda v: f"{v:.3g}"))
SXI = pd.DataFrame(rows2)
SXI["holm"] = False
for g, d in SXI.groupby("group"):
    SXI.loc[d.index, "holm"] = J.holm(d.p.values, len(d))
print("\nTreatment by sex interaction, run features (Holm per group over the 36):")
print(SXI[SXI.holm | SXI.feature.isin(["fg_pow_ca1", "fg_pow_ca3", "fg_pow_dg", "mua_ca3", "mua_dg"])]
      .to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("Significant interactions per group:", SXI.groupby("group").holm.sum().to_dict())

**Result (run part).** No run feature differs by sex in vehicle sessions after Holm over 36 (smallest p = 0.011, DG theta power); five have p < 0.05 uncorrected. Two run features have a significant sex interaction with SST+ suppression (CA3 FG frequency and CA1-CA3 theta coherence), neither of them a named exception; the named FG power and MUA interactions are not significant (p from 0.021 to 0.85). With the rest part, tables6-sex-swr-size is not reproduced (SWR size does not differ, p = 0.52) and sex-stratified-effects is not reproduced (no named exception holds).

## figs6 claims (run features)

Double suppression (PS) against PV-Cre and SST-Cre, and the empty vector group, over the 36 run features.

In [ ]:
a = res[res.reading == "a"]
tab = pd.DataFrame({
    "PS coef": a[a.test == "PS"].set_index("feature").coef, "PS p": a[a.test == "PS"].set_index("feature").p,
    "PS sig": a[a.test == "PS"].set_index("feature").significant,
    "PV vs PS p": a[a.test == "PV vs PS"].set_index("feature").p, "PV vs PS sig": a[a.test == "PV vs PS"].set_index("feature").significant,
    "PV coef": a[a.test == "PV"].set_index("feature").coef,
    "SST vs PS p": a[a.test == "SST vs PS"].set_index("feature").p, "SST vs PS sig": a[a.test == "SST vs PS"].set_index("feature").significant,
    "SST coef": a[a.test == "SST"].set_index("feature").coef,
    "EV coef": a[a.test == "EV"].set_index("feature").coef, "EV p": a[a.test == "EV"].set_index("feature").p,
    "EV sig": a[a.test == "EV"].set_index("feature").significant})
print("Reading (a):")
print(tab.to_string(float_format=lambda v: f"{v:.3g}"))
b = res[res.reading == "b"]
for t in ("PV vs PS", "SST vs PS", "EV", "PS"):
    print(f"Significant, reading (a) / (b), {t}:", a[(a.test == t) & a.significant].feature.tolist(), "/",
          b[(b.test == t) & b.significant].feature.tolist())
print("Mice: PS", S[S.group == "PS"].animal.nunique(), " EV", S[S.group == "EV"].animal.nunique())

**Result (run part).** Double suppression against PV-Cre: of 36 run features only DG theta power differs; theta modulation of SG in CA3 and DG, the claimed exceptions, does not (p = 0.60 and 0.52). Double suppression against SST-Cre: CA1 FG frequency and DG MUA differ; none of the six run features the claim lists as differing does (p from 0.28 to 0.98). The P/S treatment raises FG frequency in DG (p = 2.8e-10) but not in CA1 or CA3, and does not change CA1 FG power (p = 0.81). Empty vector mice change with CNO in seven run features (DG theta, SG and FG power among them). With the rest part: figs6-double-like-pv is not reproduced (the exceptions fail, though only one of 53 features differs), figs6-double-unlike-sst is not reproduced, and figs6-empty-vector-control is not reproduced.

## Summary

| Claim | Outcome |
|---|---|
| running-speed-unchanged | not reproduced |
| fig5b-mua-run | not reproduced |
| fig5d-sg-power-run | not reproduced |
| fig5e-sg-coherence-run | not reproduced |
| fig5f-sg-frequency-pv | not reproduced |
| fig5g-sg-theta-modulation | not reproduced |
| fig6b-fg-power-sst | not reproduced |
| figs4e-sg-fg-inverse | not reproduced |
| fig6e-sg-fg-ratio | not reproduced |
| fig6f-fg-coherence-unchanged | reproduced |
| fig6g-fg-frequency-sst | partially reproduced |
| fig6h-fg-theta-modulation-sst | not reproduced |
| figs4h-fg-theta-modulation-pv | partially reproduced |
| figs5-theta-power-ca1 | not reproduced |
| figs5-theta-coherence-frequency | not reproduced |
| tables4-genotype-baseline | partially reproduced |
| tables6-sex-swr-size | not reproduced (with the rest notebook) |
| sex-stratified-effects | not reproduced (with the rest notebook) |
| figs6-double-like-pv, figs6-double-unlike-sst, figs6-empty-vector-control | not reproduced (with the rest notebook) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's: coherence with the CA1 SWR site as the CA1 reference; frequency at one site per region (CA1 SWR site, CA3 and DG MUA sites); the 1 s bins wholly in run; the genotype test fitted from cell statistics; the log transform under two readings.

What was found about the dataset: every mouse ran its vehicle sessions before its CNO sessions, so treatment is confounded with session order, and the empty vector mice change with CNO in seven of 36 run features; the session start times are wrong (sessions of a mouse start minutes apart); sessions last 25 min, not 30.

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min")